In [323]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder

In [324]:
data = pd.read_csv('data_encoded.csv')
data.head()

,periodo,cole_area_ubicacion,cole_bilingue,cole_caracter,cole_cod_mcpio_ubicacion,cole_jornada,cole_mcpio_ubicacion,cole_naturaleza,cole_nombre_establecimiento,estu_genero,...,fami_personashogar,fami_tienecomputador,fami_tieneinternet,fami_tienelavadora,punt_ingles,punt_matematicas,punt_sociales_ciudadanas,punt_c_naturales,punt_lectura_critica,punt_global
0,20162,1,0,2,15238,1,DUITAMA,1,COLEGIO GUILLERMO LEON VALENCIA,1.0,...,NaN,NaN,NaN,NaN,61.0,59.0,63.0,57.0,56.0,295.0
1,20142,0,0,2,15759,1,SOGAMOSO,1,INSTITUCION EDUCATIVA LA INDEPENDENCIA,0.0,...,6.0,1.0,0.0,0.0,52.0,55.0,54.0,60.0,54.0,277.0
2,20152,1,0,1,15835,6,TURMEQUE,1,I.E. TECNICA INDUSTRIAL,0.0,...,3.0,0.0,0.0,1.0,55.0,52.0,50.0,53.0,47.0,254.0
3,20172,1,0,2,15176,1,CHIQUINQUIRA,1,I.E. TECNICA PIO ALBERTO FERRO PEÑA,0.0,...,5.5,0.0,0.0,1.0,59.0,58.0,46.0,51.0,45.0,253.0
4,20172,1,0,2,15759,1,SOGAMOSO,0,COLEGIO COOPERATIVO REYES PATRIA,1.0,...,3.5,1.0,1.0,1.0,75.0,66.0,70.0,65.0,64.0,335.0


In [325]:
data.columns.tolist()

['periodo',
 'cole_area_ubicacion',
 'cole_bilingue',
 'cole_caracter',
 'cole_cod_mcpio_ubicacion',
 'cole_jornada',
 'cole_mcpio_ubicacion',
 'cole_naturaleza',
 'cole_nombre_establecimiento',
 'estu_genero',
 'estu_privado_libertad',
 'fami_cuartoshogar',
 'fami_educacionmadre',
 'fami_educacionpadre',
 'fami_estratovivienda',
 'fami_personashogar',
 'fami_tienecomputador',
 'fami_tieneinternet',
 'fami_tienelavadora',
 'punt_ingles',
 'punt_matematicas',
 'punt_sociales_ciudadanas',
 'punt_c_naturales',
 'punt_lectura_critica',
 'punt_global']

¿En qué medida el acceso a tecnología en el hogar modera la brecha entre el puntaje de matemáticas y lectura crítica, y cuál es la magnitud esperada de esa brecha para un estudiante dado el perfil socioeconómico de su familia y las características de su institución en Boyacá?

In [326]:
cols_to_model = [
    # Variable objetivo 
    'punt_matematicas',
    'punt_lectura_critica',

    # Variable moderadora
    'fami_tienecomputador',
    'fami_tieneinternet',

    # Perfil socioeconómico de la familia
    'fami_estratovivienda',
    'fami_educacionmadre',
    'fami_educacionpadre',
    'fami_cuartoshogar',
    'fami_personashogar',
    'fami_tienelavadora',

    # Características de la institución
    'cole_area_ubicacion',
    'cole_bilingue',
    'cole_caracter',
    'cole_jornada',
    'cole_naturaleza',

    # Ubicación 
    'cole_cod_mcpio_ubicacion',

    # Control
    'estu_genero',
]
data = data[cols_to_model]

In [327]:
data[cols_to_model].isna().sum()

punt_matematicas               0
punt_lectura_critica           0
fami_tienecomputador        1489
fami_tieneinternet          2464
fami_estratovivienda        2642
fami_educacionmadre         2382
fami_educacionpadre         2381
fami_cuartoshogar           1541
fami_personashogar          1598
fami_tienelavadora          1491
cole_area_ubicacion            0
cole_bilingue                  0
cole_caracter                  0
cole_jornada                   0
cole_naturaleza                0
cole_cod_mcpio_ubicacion       0
estu_genero                  115
dtype: int64

In [328]:
(data[cols_to_model].isna().sum() / len(data) * 100).round(2)

punt_matematicas            0.00
punt_lectura_critica        0.00
fami_tienecomputador        1.49
fami_tieneinternet          2.46
fami_estratovivienda        2.64
fami_educacionmadre         2.38
fami_educacionpadre         2.38
fami_cuartoshogar           1.54
fami_personashogar          1.60
fami_tienelavadora          1.49
cole_area_ubicacion         0.00
cole_bilingue               0.00
cole_caracter               0.00
cole_jornada                0.00
cole_naturaleza             0.00
cole_cod_mcpio_ubicacion    0.00
estu_genero                 0.11
dtype: float64

In [329]:
print(f' porcentaje de pérdida de filas dado a eliminación: {(len(data)- len(data.dropna(subset=cols_to_model))) / len(data)*100}')

 porcentaje de pérdida de filas dado a eliminación: 3.897090425691411


In [330]:
faltantes = data[data['fami_tieneinternet'].isna()]

print(f"Faltantes vs Completo: cole_area_ubicacion")
print(faltantes['cole_area_ubicacion'].value_counts(normalize=True))
print(data['cole_area_ubicacion'].value_counts(normalize=True))

print('--'*20)
print("\nFaltantes vs Completo: cole_naturaleza")
print(faltantes['cole_naturaleza'].value_counts(normalize=True))
print(data['cole_naturaleza'].value_counts(normalize=True))

print('=='*20)
print(faltantes['cole_area_ubicacion'].value_counts(normalize=True) - data['cole_area_ubicacion'].value_counts(normalize=True))
print(faltantes['cole_naturaleza'].value_counts(normalize=True) - data['cole_naturaleza'].value_counts(normalize=True))

Faltantes vs Completo: cole_area_ubicacion
cole_area_ubicacion
1    0.852679
0    0.147321
Name: proportion, dtype: float64
cole_area_ubicacion
1    0.845056
0    0.154944
Name: proportion, dtype: float64
----------------------------------------

Faltantes vs Completo: cole_naturaleza
cole_naturaleza
1    0.815747
0    0.184253
Name: proportion, dtype: float64
cole_naturaleza
1    0.814731
0    0.185269
Name: proportion, dtype: float64
cole_area_ubicacion
1    0.007623
0   -0.007623
Name: proportion, dtype: float64
cole_naturaleza
1    0.001016
0   -0.001016
Name: proportion, dtype: float64


In [331]:
data_full_cleaned = data.dropna(subset=cols_to_model).copy()
data_full_cleaned.head(5)

,punt_matematicas,punt_lectura_critica,fami_tienecomputador,fami_tieneinternet,fami_estratovivienda,fami_educacionmadre,fami_educacionpadre,fami_cuartoshogar,fami_personashogar,fami_tienelavadora,cole_area_ubicacion,cole_bilingue,cole_caracter,cole_jornada,cole_naturaleza,cole_cod_mcpio_ubicacion,estu_genero
1,55.0,54.0,1.0,0.0,1.0,2.0,3.0,2.0,6.0,0.0,0,0,2,1,1,15759,0.0
2,52.0,47.0,0.0,0.0,1.0,2.0,2.0,2.0,3.0,1.0,1,0,1,6,1,15835,0.0
3,58.0,45.0,0.0,0.0,2.0,6.0,3.0,3.0,5.5,1.0,1,0,2,1,1,15176,0.0
4,66.0,64.0,1.0,1.0,3.0,8.0,8.0,2.0,3.5,1.0,1,0,2,1,0,15759,1.0
5,65.0,53.0,1.0,0.0,2.0,4.0,3.0,3.0,7.0,0.0,1,0,3,6,0,15759,1.0


### Hacer construccion de variables propias para el modelo

In [332]:
# brecha entre matematicas y lectura crítica. Si positivo mats>lectura, si negativo lectura>mat, si 0 igual
data_full_cleaned['brecha_mat_lec'] = data_full_cleaned['punt_matematicas'] - data_full_cleaned['punt_lectura_critica']

# indice tecnologico 
data_full_cleaned['punt_tecnologia'] = ((data_full_cleaned['fami_tienecomputador'] == 1).astype(int) + (data_full_cleaned['fami_tieneinternet'] == 1).astype(int))

data_full_cleaned.head(5)

,punt_matematicas,punt_lectura_critica,fami_tienecomputador,fami_tieneinternet,fami_estratovivienda,fami_educacionmadre,fami_educacionpadre,fami_cuartoshogar,fami_personashogar,fami_tienelavadora,cole_area_ubicacion,cole_bilingue,cole_caracter,cole_jornada,cole_naturaleza,cole_cod_mcpio_ubicacion,estu_genero,brecha_mat_lec,punt_tecnologia
1,55.0,54.0,1.0,0.0,1.0,2.0,3.0,2.0,6.0,0.0,0,0,2,1,1,15759,0.0,1.0,1
2,52.0,47.0,0.0,0.0,1.0,2.0,2.0,2.0,3.0,1.0,1,0,1,6,1,15835,0.0,5.0,0
3,58.0,45.0,0.0,0.0,2.0,6.0,3.0,3.0,5.5,1.0,1,0,2,1,1,15176,0.0,13.0,0
4,66.0,64.0,1.0,1.0,3.0,8.0,8.0,2.0,3.5,1.0,1,0,2,1,0,15759,1.0,2.0,2
5,65.0,53.0,1.0,0.0,2.0,4.0,3.0,3.0,7.0,0.0,1,0,3,6,0,15759,1.0,12.0,1


In [333]:
data_full_cleaned.columns

Index(['punt_matematicas', 'punt_lectura_critica', 'fami_tienecomputador',
       'fami_tieneinternet', 'fami_estratovivienda', 'fami_educacionmadre',
       'fami_educacionpadre', 'fami_cuartoshogar', 'fami_personashogar',
       'fami_tienelavadora', 'cole_area_ubicacion', 'cole_bilingue',
       'cole_caracter', 'cole_jornada', 'cole_naturaleza',
       'cole_cod_mcpio_ubicacion', 'estu_genero', 'brecha_mat_lec',
       'punt_tecnologia'],
      dtype='object')

In [334]:
data_full_cleaned.loc[data_full_cleaned['fami_estratovivienda'] == 0, 'fami_estratovivienda'] = np.nan
data_full_cleaned.loc[data_full_cleaned['fami_educacionmadre'] == -1, 'fami_educacionmadre'] = np.nan
data_full_cleaned.loc[data_full_cleaned['cole_bilingue'] == -1, 'cole_bilingue'] = np.nan
data_full_cleaned.loc[data_full_cleaned['cole_caracter'] == -1, 'cole_caracter'] = np.nan
data_full_cleaned.loc[data_full_cleaned['cole_caracter'] == 4, 'cole_caracter'] = np.nan

In [335]:
data_full_cleaned.drop(['fami_tienecomputador', 'estu_genero','cole_cod_mcpio_ubicacion','cole_bilingue','fami_tienelavadora', 'fami_tieneinternet','punt_matematicas', 'punt_lectura_critica'], axis=1, inplace=True)
data_full_cleaned.columns

Index(['fami_estratovivienda', 'fami_educacionmadre', 'fami_educacionpadre',
       'fami_cuartoshogar', 'fami_personashogar', 'cole_area_ubicacion',
       'cole_caracter', 'cole_jornada', 'cole_naturaleza', 'brecha_mat_lec',
       'punt_tecnologia'],
      dtype='object')

In [336]:
data_finalSQ = data_full_cleaned.dropna()
data_finalSQ.to_csv('data_finalSQ.csv', index=False)